# small128 distill-better GRID — γ-disagreement × lr × blend, step-checkpointed

**Question:** how hard can we push distillation of the (judged-good) iter-2 corpus into vh1?
Iter-2's plain run was REJECTED at 5k (mean −11%) — but its first checkpoint was at
**12,414 optimizer steps** vs gate-3's winning **~430 steps**: the absorption optimum was
plausibly never observed. This grid tests (a) checkpoint-by-steps, (b) γ-disagreement
weighting (corrections = 16.7% of main rows, 418,802 states, precomputed vs vh1),
(c) lr, (d) hard-CE share — six seeded arms, one epoch each, `--save-every-steps 500`.

**Base: `small128_vh1`** (5k bar: mean 13,080 / P50 9,323 / P5 1,222 / <1000 3.5%).
**Tensor: `iter2_mixg.pt`** = iter2_mix + `disagree_mask`.

**Upload to `MyDrive/alphatrain/`:** `colorlines_pillar3d_v4.tar.gz` (523,429 B — has
--disagree-gamma + --save-every-steps), `iter2_mixg.pt.gz` (340,635,357 B).
(`small128_vh1.pt` already on Drive.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE='/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v4.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v4.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
t0=time.time()
!cp {DRIVE}/iter2_mixg.pt.gz /content/iter2_mixg.pt.gz
gz=os.path.getsize('/content/iter2_mixg.pt.gz'); print(f'.gz: {gz:,}')
assert gz == 340_635_357, f'.gz truncated! got {gz}'
!gunzip -t /content/iter2_mixg.pt.gz && echo 'integrity OK'
!gzip -dc /content/iter2_mixg.pt.gz > /content/alphatrain/data/iter2_mixg.pt
pt=os.path.getsize('/content/alphatrain/data/iter2_mixg.pt')
assert pt == 1_016_827_575, f'.pt size wrong! got {pt}'
print(f'corpus OK ({time.time()-t0:.0f}s)')
!rm /content/iter2_mixg.pt.gz
!cp {DRIVE}/small128_vh1.pt /content/alphatrain/data/
assert os.path.getsize('/content/alphatrain/data/small128_vh1.pt') == 36_156_933
!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0); print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== ARMS (seeded; 1 epoch each; step-checkpointed every 500) =====
ARMS = [
    # name,        gamma, lr,     blend(soft share)
    ('g0_lr1',     0.0,   1e-4,   0.5),   # control = iter-2 recipe + step ckpts
    ('g2_lr1',     2.0,   1e-4,   0.5),
    ('g6_lr1',     6.0,   1e-4,   0.5),
    ('g0_lr3',     0.0,   3e-4,   0.5),
    ('g6_lr3',     6.0,   3e-4,   0.5),
    ('g2_hard',    2.0,   1e-4,   0.3),   # 70% hard-CE
]
KEEP_STEPS = [500, 1000, 2000, 4000, 8000]   # + epoch_1 (12,414 steps)
print(f'{len(ARMS)} arms')

In [ ]:
import subprocess, shutil, glob, os
%cd /content
DRIVE='/content/drive/MyDrive/alphatrain'
for name, gamma, lr, blend in ARMS:
    run = f'grid_{name}'
    print(f'===== {run}: gamma={gamma} lr={lr} blend={blend} =====', flush=True)
    cmd = f'''PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
        --tensor-file alphatrain/data/iter2_mixg.pt \
        --resume alphatrain/data/small128_vh1.pt --warm-start \
        --channels 128 --seed 42 --amp --compile \
        --epochs 1 --batch-size 4096 --lr {lr} --warmup-epochs 1 \
        --target-temperature 1.0 --decisiveness-power 0 --blend-alpha {blend} \
        --disagree-gamma {gamma} --save-every-steps 500 \
        --save-dir /content/checkpoints/{run}'''
    subprocess.run(cmd, shell=True, check=True)
    # copy the step grid + final epoch to Drive
    for s in KEEP_STEPS:
        src = f'/content/checkpoints/{run}/e1_s{s}.pt'
        if os.path.exists(src):
            shutil.copy(src, f'{DRIVE}/{run}_s{s}.pt'); print('saved', f'{run}_s{s}.pt')
    src = f'/content/checkpoints/{run}/epoch_1.pt'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{run}_s12414.pt'); print('saved', f'{run}_s12414.pt')
    shutil.rmtree(f'/content/checkpoints/{run}')   # free disk between arms
print('ALL ARMS DONE')

## Gate reads (M5, C++ eval)

36 checkpoints land on Drive as `grid_{arm}_s{steps}.pt`. Read order (each ~3 min):
1. **Sweep the control arm's step axis first** (`g0_lr1` at s500..s12414) — this alone
   answers "was iter-2's optimum inside epoch 1?"
2. Then the γ/lr/blend arms at the control's best step ± one notch.
3. Floor-first vs vh1 (P5 1,222 / P10 1,889 / <1000 3.5%, 5k figures); anything that
   clears at 500 seeds goes to the 5k for confirmation.

The grid answers: optimum step count, whether γ shifts the ceiling (adoption boost),
whether lr 3e-4 is usable under the anchor stack, and whether more hard-CE helps.
Winner ⇒ small128_vh2 + the recipe update goes into docs/small128_loop.md.